In [1]:
# ----- EXERCISE 1 -----
# ----- TF-IDF VECTORIZATION ----

import numpy as np
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from tabulate import tabulate
from collections import Counter
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Download NLTK data (run once)
nltk.download('stopwords')
nltk.download('punkt')

# Text preprocessing function
def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()
    # Remove punctuation and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Tokenize and remove stop words
    stop_words = set(stopwords.words('english'))
    words = text.split()
    words = [word for word in words if word not in stop_words]
    # Stemming
    stemmer = PorterStemmer()
    words = [stemmer.stem(word) for word in words]
    return ' '.join(words)

# Step 1: Create dataset
dataset = ["I love playing football on the weekends",
           "I enjoy hiking and camping in the mountains",
           "I like to read books and watch movies",
           "I prefer playing video games over sports",
           "I love listening to music and going to concerts"]

# Step 2: Apply preprocessing
print("Original documents:")
for doc in dataset:
    print(f"  {doc}")

print("\nPreprocessed documents:")
preprocessed_docs = [preprocess_text(doc) for doc in dataset]
for doc in preprocessed_docs:
    print(f"  {doc}")

# Step 3: Vectorize with TF-IDF
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(preprocessed_docs)

# Step 4: Perform clustering
k = 2 # Define the number of clusters
km = KMeans(n_clusters=k)
km.fit(X)

# Predict the clusters for each document
y_pred = km.predict(X)

# Display the document and its predicted cluster in a table
table_data = [["Document", "Predicted Cluster"]]
table_data.extend([[doc, cluster] for doc, cluster in zip(dataset, y_pred)])
print(tabulate(table_data, headers="firstrow"))

# Print top terms per cluster
print("\nTop terms per cluster:")
order_centroids = km.cluster_centers_.argsort()[:, ::-1]
terms = vectorizer.get_feature_names_out()
for i in range(k):
    print("Cluster %d:" % i)
    for ind in order_centroids[i, :10]:
        print(' %s' % terms[ind])
        print()

#Evaluate result
# Calculate purity
total_samples = len(y_pred)
cluster_label_counts = [Counter(y_pred)]
purity = sum(max(cluster.values()) for cluster in cluster_label_counts) / total_samples
print("Purity:", purity)

Original documents:
  I love playing football on the weekends
  I enjoy hiking and camping in the mountains
  I like to read books and watch movies
  I prefer playing video games over sports
  I love listening to music and going to concerts

Preprocessed documents:
  love play footbal weekend
  enjoy hike camp mountain
  like read book watch movi
  prefer play video game sport
  love listen music go concert
Document                                           Predicted Cluster
-----------------------------------------------  -------------------
I love playing football on the weekends                            0
I enjoy hiking and camping in the mountains                        0
I like to read books and watch movies                              0
I prefer playing video games over sports                           0
I love listening to music and going to concerts                    1

Top terms per cluster:
Cluster 0:
 play

 footbal

 weekend

 mountain

 hike

 enjoy

 camp

 sport

 vi

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/4af25032-c966-47e7-803b-
[nltk_data]     397ac7e4a5e7/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /home/4af25032-c966-47e7-803b-
[nltk_data]     397ac7e4a5e7/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/joblib/externals/loky/backend/context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
found 0 physical cores < 1
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/joblib/externals/loky/backend/context.py", line 255, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


In [9]:
# ----- WORD2VEC VECTORIZATION -----

# 1. Import the libraries

from gensim.models import Word2Vec

# Step 2: Tokenize preprocessed documents
tokenized_preprocessed = [doc.split() for doc in preprocessed_docs]

# 3. Train Word2Vec model
tokenized_dataset = [doc.split() for doc in dataset]
word2vec_model = Word2Vec(sentences=tokenized_dataset, vector_size=100,
window=5, min_count=1, workers=4)

# 4. Create document embeddings
def get_doc_embedding(doc_tokens, model):
    vectors = [model.wv[word] for word in doc_tokens if word in model.wv]
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(model.vector_size)

X_w2v = np.array([get_doc_embedding(doc.split(), word2vec_model) for doc in preprocessed_docs])

# Tabulate the document and predicted cluster
km_w2v = KMeans(n_clusters=k, random_state=42)
km_w2v.fit(X_w2v)
y_pred_w2v = km_w2v.predict(X_w2v)

# Step 5: Display results
print("WORD2VEC WITH PREPROCESSING RESULTS")
print("="*60)
table_data_w2v = [["Document", "Predicted Cluster"]]
for doc, cluster in zip(dataset, y_pred_w2v):
    table_data_w2v.append([doc[:50] + "..." if len(doc) > 50 else doc, cluster])
print(tabulate(table_data_w2v, headers="firstrow"))

# 6. Evaluate results
# Calculate purity
cluster_label_counts_w2v = [Counter(y_pred_w2v)]
purity_w2v = sum(max(cluster.values()) for cluster in cluster_label_counts_w2v) / total_samples
print(f"\nPurity: {purity_w2v}")

print("\nCOMPARISON: WITH vs WITHOUT PREPROCESSING")
print("="*60)
print(f"TF-IDF without preprocessing:           Purity = 0.6")
print(f"TF-IDF with preprocessing:              Purity = {purity}")
print(f"Word2Vec without preprocessing          Purity = 0.6")
print(f"Word2Vec with preprocessing:            Purity = {purity_w2v}")

WORD2VEC WITH PREPROCESSING RESULTS
Document                                           Predicted Cluster
-----------------------------------------------  -------------------
I love playing football on the weekends                            1
I enjoy hiking and camping in the mountains                        0
I like to read books and watch movies                              1
I prefer playing video games over sports                           1
I love listening to music and going to concerts                    1

Purity: 0.8

COMPARISON: WITH vs WITHOUT PREPROCESSING
TF-IDF without preprocessing:           Purity = 0.6
TF-IDF with preprocessing:              Purity = 0.8
Word2Vec without preprocessing          Purity = 0.6
Word2Vec with preprocessing:            Purity = 0.8


In [12]:
# ----- EXERCISE 2 ----
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from tabulate import tabulate
from collections import Counter
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Load the dataset
df = pd.read_csv('customer_complaints_1.csv')

# Extract text column
complaints = df['text'].fillna('').tolist()

# Preprocessing function
def preprocess_complaint(text):
    # Convert to lowercase
    text = text.lower()
    # Remove punctuation and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Tokenize and remove stop words
    stop_words = set(stopwords.words('english'))
    words = text.split()
    words = [word for word in words if word not in stop_words]
    # Stemming
    stemmer = PorterStemmer()
    words = [stemmer.stem(word) for word in words]
    return ' '.join(words)

preprocessed_complaints = [preprocess_complaint(text) for text in complaints]

# TF-IDF Vectorization
print("TF-IDF VECTORIZATION")
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(preprocessed_complaints)

# Perform clustering
k = 2  # Define the number of clusters
km = KMeans(n_clusters=k, random_state=42)
km.fit(X)

# Predict the clusters for each document
cluster_labels = km.predict(X)

# Add cluster labels to dataframe
df['cluster'] = cluster_labels

# Display results in table
table_data = [["Document (first 50 chars)", "Predicted Cluster"]]
for idx, (doc, cluster) in enumerate(zip(complaints, cluster_labels)):
    short_doc = doc[:50] + "..." if len(doc) > 50 else doc
    table_data.append([short_doc, cluster])
print(tabulate(table_data, headers="firstrow"))

# Print top terms per cluster
print("\nTop terms per cluster:")
order_centroids = km.cluster_centers_.argsort()[:, ::-1]
terms = vectorizer.get_feature_names_out()
for i in range(k):
    print(f"Cluster {i}:")
    top_terms = []
    for ind in order_centroids[i, :10]:
        top_terms.append(terms[ind])
    print(f"  {', '.join(top_terms)}")
    print()

# Calculate and print purity
total_samples = len(cluster_labels)
cluster_label_counts = [Counter(cluster_labels)]
purity = sum(max(cluster.values()) for cluster in cluster_label_counts) / total_samples
print(f"Purity: {purity}")

# Save clustered results
df.to_csv('customer_complaints_clustered.csv', index=False)
print("\nClustered data saved to 'customer_complaints_clustered.csv'")

TF-IDF VECTORIZATION
Document (first 50 chars)                                Predicted Cluster
-----------------------------------------------------  -------------------
I used to love Comcast. Until all these constant u...                    1
I'm so over Comcast! The worst internet provider. ...                    1
If I could give them a negative star or no stars o...                    0
I've had the worst experiences so far since instal...                    0
Check your contract when you sign up for Comcast a...                    0
Thank God. I am changing to Dish. They gave me awe...                    1
I Have been a long time customer and only have Xfi...                    1
There is a malfunction on the DVR manager which is...                    0
Charges overwhelming. Comcast service rep was so i...                    1
I have had cable, DISH, and U-verse, etc. in the p...                    1
Had them from 2014 to now. I'd tell new customers ...                    1
Disa